<a href="https://colab.research.google.com/github/Tisiaka1/Cell-Tracking-Lineage-Algorithm-/blob/main/Copy_of_BioHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Augmentation



In [ ]:
import monai
import numpy as np
from monai.transforms import(
Compose,
RandRotated,
RandZoomd,
Rand3DElasticd,
NormalizeIntensityd,
RandRotate90d



)





Transform = Compose([
    RandZoomd(
        keys = ["image","label"],
        min_zoom = 0.90 , max_zoom = 1.10,
        prob = 1.0,
        mode = ["trilinear","trilinear"]
    ),
    RandRotate90d(
        keys=["image", "label"],
        prob=0.5,
        spatial_axes=(1, 2)),

    NormalizeIntensityd(
    keys=["image"],
    nonzero=True)


])

# Ground Truth Heatmap

In [ ]:
from scipy.ndimage import gaussian_filter

#Heatmaps are goated because it allows the model to understand how far of it is from a cell
# think of 1.0 being the cell and 0.1 indicating to the model that it's far from the cell and obviously 0 is the background voxels
# think about taking an exam or something, you didn't get the right answer but the teacher gave you some credit, this lets you know you are on the right track

prob = np.zeros((9,17,17))
prob[4, 8, 8] = 1.0
PEAK = gaussian_filter(prob, sigma=(1, 2, 2)).max()
print(PEAK)
#sigma=(1, 2, 2), margin=3
#  padded_shape = (shape[0] + 2 * margin, shape[1], shape[2])
def create_heatmap(shape, x, y, z):


    heat = np.zeros(shape, dtype=np.float32)


    for z1, y1, x1 in zip(z, y, x):
        iz, iy, ix = int(round(z1)), int(round(y1)), int(round(x1))
        if 0 <= iz < shape[0] and 0 <= iy < shape[1] and 0 <= ix < shape[2]:
            heat[iz, iy, ix] = 1.0





    return  heat









0.01587401455920438


# Lazy Loader



In [ ]:
import zarr

import numpy as np
import torch
from torch.utils.data import (Dataset,DataLoader)
import random
import os



Nth_Val = 38



train_dir = "/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/train"


sample_ids = sorted(  list({
    f.split('.')[0] for f in os.listdir(train_dir) if not f.startswith('.')
}))
val_ids = set(sample_ids[:Nth_Val])
Train_ids = set(sample_ids[Nth_Val:])

val_chunks = []
train_chunks = []


for s in sample_ids:

    zarr_path = os.path.join(train_dir, f"{s}.zarr")

    Gt_path = os.path.join(train_dir, f"{s}.geff", 'nodes', 'props')


    if not os.path.exists(zarr_path):
        print(f"Warning: Zarr data directory not found for sample {s} at {zarr_path}. Skipping this sample.")
        continue


    if not os.path.exists(os.path.join(train_dir, f"{s}.geff")):
        print(f"Warning:  directory not found for sample {s} at {os.path.join(train_dir, f'{s}.geff')}. Skipping this sample.")
        continue


    data = zarr.open(zarr_path, mode='r')['0']

    num_t = data.shape[0]
    num_z = data.shape[1]




    for t in range(num_t):
        for z_start in range(0,num_z- 15,16):
            if s in val_ids :
                 val_chunks.append((zarr_path, Gt_path, t, z_start))
            else:
                train_chunks.append((zarr_path, Gt_path, t, z_start))

 #function calls the raw data
def load_image_chunk(zarr_path , t , z_start, z_step = 16, channel = None, dtype=np.float32):
        data = zarr.open(zarr_path, mode='r')['0']

        z_end = z_start + z_step
        image_chunk = data[t,z_start:z_end]

        return image_chunk


#function here calls the label

def load_gt_chunk(gt_path, t_target, z_start, z_step=16, margin=4):
    z_end = z_start + z_step

    # Open Zarr array on-the-fly instead of using RAM cache
    gt = zarr.open(gt_path, mode='r')
    t_vals = gt["t/values"][:]
    z_vals = gt["z/values"][:]
    y_vals = gt["y/values"][:]
    x_vals = gt["x/values"][:]


    mask = (t_vals == t_target) & (z_vals >= z_start - margin) & (z_vals < z_end + margin)

    return {
        "X": x_vals[mask],
        "Y": y_vals[mask],
        "Z": z_vals[mask] - z_start,   # slice number inside the chunk; can be a bit below 0 or above 15
    }


 #create an array with the desire path to raw data,label, timepoint and Depth






class Custom(Dataset):
    def __init__(self, data, transform=None, target_transform=None):
        self.data = data

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):
        zarr_path,Gt_path,t,z_start = self.data[idx]

        image_chunk = load_image_chunk(zarr_path,t,z_start)
        gt_chunk = load_gt_chunk(Gt_path, t, z_start)

        label_chunk = create_heatmap(
            shape = image_chunk.shape,
            x = gt_chunk["X"],
            y = gt_chunk["Y"],
            z = gt_chunk["Z"]
        )


        images =  image_chunk[None]
        labels =  label_chunk[None]

        data_dict = {"image":images,"label":labels}
        Transform_data = Transform(data_dict)

        Raw_tensor = Transform_data["image"].float()
        Label_tensor = Transform_data["label"].float()



        return Raw_tensor, Label_tensor #banGGGGG



train_loader = Custom( data = train_chunks, transform = Transform)
val_loader = Custom( data = val_chunks, transform = Transform )


train_loader = DataLoader(
    train_loader,
    batch_size = 2,
    shuffle = True,
    num_workers = 4,
    pin_memory = False,
    persistent_workers=False,
)

val_loader = DataLoader(
    val_loader,
    batch_size=2,
    shuffle=False, num_workers=2, pin_memory=False, persistent_workers=False)



# 3D UNets

In [ ]:
import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels,kernel_size=3,padding=1)
        self.norm1 = nn.InstanceNorm3d(out_channels)
        self.relu1 = nn.ReLU()
        self.conv2 = nn.Conv3d(out_channels,out_channels,kernel_size=3,padding=1)
        self.norm2 = nn.InstanceNorm3d(out_channels)
        self.relu2 = nn.ReLU()

    def forward(self,x):
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.relu1(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.relu2(x)

        return x




class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        #Encoder

        self.enc1 = ConvBlock(1,16)
        self.enc2 = ConvBlock(16,32 )
        self.pool = nn.MaxPool3d(kernel_size=(1,2,2))
        self.bottleneck = ConvBlock(32, 64)
        # decoder
        self.up = nn.Upsample(scale_factor=(1, 2, 2), mode='trilinear', align_corners=False)
        self.dec2 = ConvBlock(96, 32)
        self.dec1 = ConvBlock(48, 16)
        # output
        self.final = nn.Conv3d(16, 1, kernel_size=1)

    def forward(self, x):

        skip1 = self.enc1(x);
        x = self.pool(skip1);
        skip2 = self.enc2(x);
        x = self.pool(skip2);
        x = self.bottleneck(x);
        x = self.up(x);
        x = torch.cat([x, skip2], dim=1);
        x = self.dec2(x);
        x = self.up(x);
        x = torch.cat([x, skip1], dim=1);
        x = self.dec1(x);
        x = self.final(x);
        return x




# Loss Function

In [ ]:
import torch.nn as nn

class AdaptiveWingLoss(nn.Module):
    def __init__(self, omega=14, theta=0.5, eps=1.0, alpha=2.1, w_boost=10.0, boost_thresh=0.2):
        super().__init__()
        self.omega, self.theta, self.eps, self.alpha = omega, theta, eps, alpha
        self.w_boost, self.boost_thresh = w_boost, boost_thresh

    def forward(self, pred, target):
        e = (target - pred).abs()
        p = self.alpha - target
        A = self.omega * (1 / (1 + (self.theta / self.eps) ** p)) * p \
            * (self.theta / self.eps) ** (p - 1) / self.eps
        C = self.theta * A - self.omega * torch.log1p((self.theta / self.eps) ** p)

        small = self.omega * torch.log1p((e / self.eps) ** p)
        large = A * e - C
        loss = torch.where(e < self.theta, small, large)

        weight = torch.where(target > self.boost_thresh,
                             torch.full_like(target, self.w_boost),
                             torch.ones_like(target))
        return (loss * weight).mean()



# Training

In [ ]:
import torch
from monai.networks.layers.simplelayers import GaussianFilter

import torch.nn.functional as F

import os
import gc


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = torch.amp.GradScaler('cuda')

model = UNet().to(device)
awing = AdaptiveWingLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
gpu_blur = GaussianFilter(spatial_dims=3, sigma=(1, 2, 2)).to(device)


if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = nn.DataParallel(model)

MARGIN = 3
SIGMA = (1, 2, 2)
DEPTH_DIM_SIZE = 16
VAL_EVERY = 32000
step = 0

def validate(model,val_loader):
    model.eval()
    total_loss,total_chunks = 0.0,0
    with torch.no_grad():
         for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device)
            labels = labels.float()
            labels = F.pad(labels, (0, 0, 0, 0, MARGIN, MARGIN), mode='constant', value=0.0)
            labels = gpu_blur(labels)
            labels = labels[:, :, MARGIN : MARGIN + DEPTH_DIM_SIZE]
            labels = torch.clamp(labels / PEAK, 0.0, 1.0)


            with torch.amp.autocast('cuda'):
                preds = model(images)
                loss = awing(preds, labels)
            total_loss += loss.item() * images.shape[0]
            total_chunks += images.shape[0]
    model.train()                                 # back to training mode
    return total_loss / total_chunks




for epoch in range(5):
    model.train()
    for images, labels in train_loader :
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)


        with torch.no_grad():
            labels = labels.float()
            labels = F.pad(labels, (0, 0, 0, 0, MARGIN, MARGIN), mode='constant', value=0.0)
            labels = gpu_blur(labels)
            labels = labels[:, :, MARGIN : MARGIN + DEPTH_DIM_SIZE]
            labels = torch.clamp(labels / PEAK, 0.0, 1.0)

        optimizer.zero_grad()


        with torch.amp.autocast('cuda'):
            preds =  model(images)
            loss = awing(preds,labels)


        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        step += 1
        if step % 1000 == 0:
            print(f"epoch {epoch} | step {step} | train loss {loss.item():.4f} | pred max {preds.max().item():.3f}")






    print(f"Epoch {epoch} finished.")
    gc.collect()

epoch 0 | step 10000 | train loss 0.0026 | pred max 0.455
epoch 0 | step 20000 | train loss 0.0014 | pred max 0.177
epoch 0 | step 30000 | train loss 0.0043 | pred max 0.157
Epoch 0 finished.
epoch 1 | step 40000 | train loss 0.0074 | pred max 0.601
epoch 1 | step 50000 | train loss 0.0005 | pred max 0.645
epoch 1 | step 60000 | train loss 0.0001 | pred max 0.256
Epoch 1 finished.
epoch 2 | step 70000 | train loss 0.0001 | pred max 0.279
epoch 2 | step 80000 | train loss 0.0012 | pred max 0.531
epoch 2 | step 90000 | train loss 0.0061 | pred max 0.286
Epoch 2 finished.
epoch 3 | step 100000 | train loss 0.0058 | pred max 0.492
epoch 3 | step 110000 | train loss 0.0005 | pred max 0.341
epoch 3 | step 120000 | train loss 0.0008 | pred max 0.493
Epoch 3 finished.
epoch 4 | step 130000 | train loss 0.0008 | pred max 0.509
epoch 4 | step 140000 | train loss 0.0031 | pred max 0.722
epoch 4 | step 150000 | train loss 0.0016 | pred max 0.527
epoch 4 | step 160000 | train loss 0.0029 | pred max

In [ ]:
import torch
import os

save_dir = '/content/drive/MyDrive/biohub_data/models'
os.makedirs(save_dir, exist_ok=True)
model_save_path = os.path.join(save_dir, '3d_unet_biohub.pth')

torch.save(model.state_dict(), model_save_path)
print(f"Model successfully saved to {model_save_path}")

NameError: name 'model' is not defined

# Testing

In [ ]:
#sample_ids = sorted(  list({
    #f.split('.')[0] for f in os.listdir(train_dir) if not f.startswith('.')
#}))
Submission_path = '/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/Submission.csv'
from monai.transforms import(Compose,NormalizeIntensityd)
test_transforms = Compose([
    NormalizeIntensityd(keys=["image"], nonzero=True)
])
import zarr

import numpy as np
import torch
from torch.utils.data import (Dataset,DataLoader)
import random
import os



class TestDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        zarr_path, t, z_start = self.data[idx]
        try:
          image_chunk = load_image_chunk(zarr_path, t, z_start)
        except Exception as e:
          print(f"\n⚠️ CORRUPT CHUNK DETECTED in: {zarr_path} at t={t}, z={z_start}")
          i



        images = image_chunk[None]
        data_dict = {"image": images}
        metaData = (zarr_path,t,z_start)

        if self.transform:
            data_dict = self.transform(data_dict)

        Raw_tensor = torch.from_numpy(data_dict["image"]).float()
        return Raw_tensor, metaData

test_dir = "/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test"
test_chunks = []
test_ids = sorted(list({
    f.split('.')[0] for f in os.listdir(test_dir) if not f.startswith('.')
}))


for s in test_ids:
    zarr_path = os.path.join(test_dir, f"{s}.zarr")

    if not os.path.exists(zarr_path):
        print(f"Warning: Zarr data directory not found for sample {s} at {zarr_path}. Skipping.")
        continue

    data = zarr.open(zarr_path, mode='r')['0']
    num_t = data.shape[0]
    num_z = data.shape[1]
    print(s)
    for t in range(num_t):
        for z_start in range(0, num_z - 15, 16):
            test_chunks.append((zarr_path, t, z_start))




test_loader = TestDataset( data = test_chunks)

test_loader = DataLoader(
     test_loader,
     batch_size = 4,
     shuffle = False,
     num_workers = 4,
     pin_memory = False,
     persistent_workers=False,
 )



print(test_chunks)




44b6_0113de3b
44b6_0b24845f
6bba_05b6850b
6bba_05db0fb1
[('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 0, 0), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 0, 16), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 0, 32), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 0, 48), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 1, 0), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 1, 16), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 1, 32), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr', 1, 48), ('/content/drive/MyDrive/biohub_data/biohub-cell-tracking

In [ ]:
from skimage.feature import peak_local_max
from skimage.draw import ellipsoid
import numpy as np
import pandas as pd
from pathlib import Path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)


Unet_path = '/content/drive/MyDrive/biohub_data/models/3d_unet_biohub.pth'


df = pd.read_csv(Submission_path )

model.load_state_dict(torch.load(Unet_path))


id = 0

model.eval()
footprint = ellipsoid(4,17,17)
row_id = 0
node_id_tracker = {}
submissions_row = []
with torch.no_grad():

    for batch_idx,(images,metaData) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        preds = model(images)

        print(f"⏳ Processing batch {batch_idx + 1} of {len(test_loader)}...")
        preds_np = preds.cpu().numpy()

        zarr_path,times,z_start = metaData[0],metaData[1],metaData[2]

        for i in range(images.size(0)):

          current_zarr = zarr_path[i]
          current_zarr = Path(current_zarr).stem
          current_t = times[i].item()
          Current_z_start = z_start[i].item()

          if current_zarr not in node_id_tracker:
            node_id_tracker[current_zarr] = 0

          single_3d_heatmap = preds_np[i,0]

          coordinates = peak_local_max(single_3d_heatmap, footprint=footprint)
          for coord in coordinates :


            node_id_tracker[current_zarr] +=1
            current_node_id = node_id_tracker[current_zarr]
            z_local = coord[0]
            y = coord[1]
            x = coord[2]



            submissions_row.append({
                    'id': row_id,
                    'dataset': current_zarr,
                    'row_type': 'node',
                    'node_id': current_node_id,
                    't': current_t,
                    'z': z_local,
                    'y': y,
                    'x': x,
                    'source_id': -1,
                    'target_id': -1
                })
          row_id += 1


final_df = pd.DataFrame(submissions_row)
final_df.to_csv(Submission_path, index=False)
print("Inference complete and submission saved!")

















⏳ Processing batch 1 of 400...
⏳ Processing batch 2 of 400...
⏳ Processing batch 3 of 400...
⏳ Processing batch 4 of 400...
⏳ Processing batch 5 of 400...
⏳ Processing batch 6 of 400...
⏳ Processing batch 7 of 400...
⏳ Processing batch 8 of 400...
⏳ Processing batch 9 of 400...
⏳ Processing batch 10 of 400...
⏳ Processing batch 11 of 400...

⚠️ CORRUPT CHUNK DETECTED in: /content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr at t=19, z=0

⚠️ CORRUPT CHUNK DETECTED in: /content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr at t=19, z=16

⚠️ CORRUPT CHUNK DETECTED in: /content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr at t=19, z=32

⚠️ CORRUPT CHUNK DETECTED in: /content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/test/44b6_0113de3b.zarr at t=19, z=48
⏳ Processing batch 12 of 400...
⏳ Processing batch 13 of 400...
⏳ Processing batch

# Tracking algorithm


In [ ]:
import numpy as np
from lap import lapjv # According to gemini and resources online the Jonker-volgenant algorithm is faster than the hungarian algorithm for solving a linear assignment problem.
from scipy.spatial.distance import cdist

from google.colab import files
df = pd.DataFrame(submissions_row)
my_edges = []
unique_zarr_file = df['dataset'].unique()
for current_zarr in unique_zarr_file:
    current_zarr_df = df[df['dataset'] == current_zarr]
    max_time = current_zarr_df['t'].max()

    for current_t in range(max_time):
       cells_t0 = df[(df['dataset']==current_zarr) & (df['t']==current_t)]
       cells_t1 = df[(df['dataset']==current_zarr) & (df['t']==current_t+1)]

       coords_t0 = cells_t0[['z','y','x']].to_numpy()
       coords_t1 = cells_t1[['z','y','x']].to_numpy()
       ids_t0 = cells_t0['node_id'].tolist()
       ids_t1 = cells_t1['node_id'].tolist()

       cost_matrix = cdist(coords_t0,coords_t1,metric='euclidean')
       cost_matrix = cost_matrix.astype(np.float64)

       total_cost,row_assign,col_assign = lapjv(cost_matrix,extend_cost=True)


       #assuming if a cell move more than 15 pixel its probably a false match and it died.

       MAX_DISTANCE = 15.0

       for row_idx, col_idx in enumerate(row_assign):
        if col_idx == -1:
          continue

        distance = cost_matrix[row_idx,col_idx]

        if distance <= MAX_DISTANCE:
          Source_node = ids_t0[row_idx]
          target_node = ids_t1[col_idx]
          my_edges.append({
                                'dataset': current_zarr,
                                'row_type': 'edge',
                                'node_id': -1,   # Edges don't get their own ID
                                't': -1,         # Edges don't have a time
                                'z': -1, 'y': -1, 'x': -1,
                                'source_id': Source_node,
                                'target_id': target_node
                            })


edges_df = pd.DataFrame(my_edges)
print(edges_df.head())
final_submission = pd.concat([df, edges_df], ignore_index=True)


final_submission.to_csv('/content/drive/MyDrive/biohub_data/biohub-cell-tracking-during-development/Submission.csv', index=False)

final_submission.to_csv('guaranteed_submission.csv', index=False)

print("Forcing download to your computer...")

files.download('guaranteed_submission.csv')
print("Tracking complete! Edges successfully added.")


         dataset row_type  node_id  t  z  y  x  source_id  target_id
0  44b6_0113de3b     edge       -1 -1 -1 -1 -1          1        326
1  44b6_0113de3b     edge       -1 -1 -1 -1 -1          2        325
2  44b6_0113de3b     edge       -1 -1 -1 -1 -1          3        339
3  44b6_0113de3b     edge       -1 -1 -1 -1 -1          4        330
4  44b6_0113de3b     edge       -1 -1 -1 -1 -1          5        623
Forcing download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Tracking complete! Edges successfully added.


In [ ]:

print(f"1. Items in submission_rows list: {len(submissions_row)}")


edges_df = pd.DataFrame(submissions_row)
if not edges_df.empty:
    edges_in_df = len(edges_df[edges_df['row_type'] == 'edge'])
    print(f"2. Edges successfully converted to DataFrame: {edges_in_df}")
else:
    print("2. FAILED: edges_df is empty!")


final_submission = pd.concat([df, edges_df], ignore_index=True)
edges_in_final = len(final_submission[final_submission['row_type'] == 'edge'])


edges_df.to_csv('just_edges_test.csv', index=False)


1. Items in submission_rows list: 151160
2. Edges successfully converted to DataFrame: 0
3. Edges surviving after pd.concat: 0

Saved 'just_edges_test.csv' locally.


{'id': 0, 'dataset': '44b6_0113de3b', 'row_type': 'node', 'node_id': 1, 't': 0, 'z': np.int64(8), 'y': np.int64(37), 'x': np.int64(77), 'source_id': -1, 'target_id': -1}
